In [2]:
# 计算 test_grid.ipynb 的“训练监督(render_3d) vs 输出图”LPIPS
import os
import re
import glob
import sys
import subprocess
from pathlib import Path

import cv2
import numpy as np
import torch

# 确保 lpips 可用（当前 kernel 就地安装）
try:
    import lpips
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lpips==0.1.4", "-q"])
    import lpips

# ===== 路径 =====
GT_DIR = "/data/test_grid4_from_test_grid4/render_3d"  # 训练监督图目录
OUT_DIR = "/home/viplab/liukejia/project/mv_model/results_grid4"  # 输出图目录
TARGET_ANGLES = [0, 90, 180, 270]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def parse_name_tokens(stem: str):
    # 文件名格式: num1_num2_num3_num4
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    try:
        nums = [int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])]
    except ValueError:
        return None
    num123 = f"{nums[0]}_{nums[1]}_{nums[2]}"
    num4 = nums[3]
    return num123, num4


def group_gt_files(gt_dir: str):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        files.extend(glob.glob(os.path.join(gt_dir, ext)))
    files = sorted(files)

    groups = {}
    for p in files:
        parsed = parse_name_tokens(Path(p).stem)
        if parsed is None:
            continue
        num123, num4 = parsed
        groups.setdefault(num123, []).append((num4, p))

    final = {}
    for num123, items in groups.items():
        angle_to_path = {}
        for num4, p in sorted(items, key=lambda x: (x[0], x[1])):
            if num4 in TARGET_ANGLES and num4 not in angle_to_path:
                angle_to_path[num4] = p
        if all(a in angle_to_path for a in TARGET_ANGLES):
            final[num123] = [angle_to_path[a] for a in TARGET_ANGLES]
    return final


def stitch_2x2_rgb(paths, image_size=512):
    # 顺序: [0,90,180,270] -> tl,tr,bl,br
    sub = image_size // 2
    imgs = []
    for p in paths:
        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (sub, sub), interpolation=cv2.INTER_AREA)
        imgs.append(rgb)

    top = np.concatenate([imgs[0], imgs[1]], axis=1)
    bottom = np.concatenate([imgs[2], imgs[3]], axis=1)
    return np.concatenate([top, bottom], axis=0)


def parse_num123_from_output_name(path: str):
    # num123_1_2_3_0001.png -> 1_2_3
    name = Path(path).stem
    m = re.match(r"num123_(-?\d+_-?\d+_-?\d+)_\d+", name)
    return None if m is None else m.group(1)


def to_lpips_tensor(rgb_uint8: np.ndarray):
    # HWC [0,255] -> 1x3xHxW [-1,1]
    t = torch.from_numpy(rgb_uint8).float() / 255.0
    t = t.permute(2, 0, 1).unsqueeze(0)
    t = t * 2.0 - 1.0
    return t


if not os.path.isdir(GT_DIR):
    print(f"未找到 GT_DIR: {GT_DIR}")
elif not os.path.isdir(OUT_DIR):
    print(f"未找到 OUT_DIR: {OUT_DIR}")
else:
    loss_fn = lpips.LPIPS(net="alex").to(DEVICE)
    loss_fn.eval()

    grouped_gt = group_gt_files(GT_DIR)
    outs = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

    values = []
    skipped = 0

    with torch.inference_mode():
        for op in outs:
            num123 = parse_num123_from_output_name(op)
            if num123 is None or num123 not in grouped_gt:
                skipped += 1
                continue

            gt_rgb = stitch_2x2_rgb(grouped_gt[num123], image_size=512)
            if gt_rgb is None:
                skipped += 1
                continue

            out_bgr = cv2.imread(op, cv2.IMREAD_COLOR)
            if out_bgr is None:
                skipped += 1
                continue
            out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
            out_rgb = cv2.resize(out_rgb, (gt_rgb.shape[1], gt_rgb.shape[0]), interpolation=cv2.INTER_AREA)

            gt_t = to_lpips_tensor(gt_rgb).to(DEVICE)
            out_t = to_lpips_tensor(out_rgb).to(DEVICE)
            v = loss_fn(gt_t, out_t).item()
            values.append((op, float(v)))

    if len(values) == 0:
        print("没有可计算的配对样本，请检查 GT_DIR 文件名是否包含 num1_num2_num3_num4。")
    else:
        lpips_list = [x[1] for x in values]
        print(f"有效样本数: {len(lpips_list)}，跳过: {skipped}")
        print(f"平均 LPIPS: {np.mean(lpips_list):.4f} (越低越好)")
        print(f"中位数 LPIPS: {np.median(lpips_list):.4f} (越低越好)")
        print("前 5 个样本:")
        for p, v in values[:5]:
            print(f"  {Path(p).name}: {v:.4f}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
有效样本数: 9，跳过: 0
平均 LPIPS: 0.3136 (越低越好)
中位数 LPIPS: 0.2956 (越低越好)
前 5 个样本:
  num123_0_0_0_0001.png: 0.3833
  num123_1_0_0_0002.png: 0.2515
  num123_2_0_0_0003.png: 0.2489
  num123_3_0_0_0004.png: 0.2922
  num123_4_0_0_0005.png: 0.3299


In [3]:
# 计算 test_grid.ipynb 的“训练监督(render_3d) vs 输出图”FID
import os
import re
import glob
import sys
import subprocess
from pathlib import Path

import cv2
import numpy as np
import torch

# 确保 FID 依赖可用
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchmetrics", "torch-fidelity", "-q"])
    from torchmetrics.image.fid import FrechetInceptionDistance

GT_DIR = "/data/test_grid4_from_test_grid4/render_3d"
OUT_DIR = "/home/viplab/liukejia/project/mv_model/results_grid4"
TARGET_ANGLES = [0, 90, 180, 270]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def parse_name_tokens(stem: str):
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    try:
        nums = [int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])]
    except ValueError:
        return None
    num123 = f"{nums[0]}_{nums[1]}_{nums[2]}"
    return num123, nums[3]

def group_gt_files(gt_dir: str):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        files.extend(glob.glob(os.path.join(gt_dir, ext)))
    files = sorted(files)

    groups = {}
    for p in files:
        parsed = parse_name_tokens(Path(p).stem)
        if parsed is None:
            continue
        num123, num4 = parsed
        groups.setdefault(num123, []).append((num4, p))

    final = {}
    for num123, items in groups.items():
        angle_to_path = {}
        for num4, p in sorted(items, key=lambda x: (x[0], x[1])):
            if num4 in TARGET_ANGLES and num4 not in angle_to_path:
                angle_to_path[num4] = p
        if all(a in angle_to_path for a in TARGET_ANGLES):
            final[num123] = [angle_to_path[a] for a in TARGET_ANGLES]
    return final

def stitch_2x2_rgb(paths, image_size=512):
    sub = image_size // 2
    imgs = []
    for p in paths:
        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (sub, sub), interpolation=cv2.INTER_AREA)
        imgs.append(rgb)

    top = np.concatenate([imgs[0], imgs[1]], axis=1)
    bottom = np.concatenate([imgs[2], imgs[3]], axis=1)
    return np.concatenate([top, bottom], axis=0)

def parse_num123_from_output_name(path: str):
    name = Path(path).stem
    m = re.match(r"num123_(-?\d+_-?\d+_-?\d+)_\d+", name)
    return None if m is None else m.group(1)

def hwc_uint8_to_nchw_uint8(img: np.ndarray):
    t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).contiguous()
    return t.to(torch.uint8)

if not os.path.isdir(GT_DIR):
    print(f"未找到 GT_DIR: {GT_DIR}")
elif not os.path.isdir(OUT_DIR):
    print(f"未找到 OUT_DIR: {OUT_DIR}")
else:
    grouped_gt = group_gt_files(GT_DIR)
    outs = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

    pairs = []
    skipped = 0
    for op in outs:
        num123 = parse_num123_from_output_name(op)
        if num123 is None or num123 not in grouped_gt:
            skipped += 1
            continue

        gt_rgb = stitch_2x2_rgb(grouped_gt[num123], image_size=512)
        if gt_rgb is None:
            skipped += 1
            continue

        out_bgr = cv2.imread(op, cv2.IMREAD_COLOR)
        if out_bgr is None:
            skipped += 1
            continue

        out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
        out_rgb = cv2.resize(out_rgb, (gt_rgb.shape[1], gt_rgb.shape[0]), interpolation=cv2.INTER_AREA)
        pairs.append((gt_rgb, out_rgb, op))

    if len(pairs) < 2:
        print(f"可用样本不足以稳定估计 FID，当前有效: {len(pairs)}，跳过: {skipped}")
    else:
        fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)

        for gt_rgb, out_rgb, _ in pairs:
            fid.update(hwc_uint8_to_nchw_uint8(gt_rgb).to(DEVICE), real=True)
            fid.update(hwc_uint8_to_nchw_uint8(out_rgb).to(DEVICE), real=False)

        fid_value = float(fid.compute().item())
        print(f"有效样本数: {len(pairs)}，跳过: {skipped}")
        print(f"FID: {fid_value:.4f} (越低越好)")

/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /home/viplab/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [04:49<00:00, 330kB/s]


有效样本数: 9，跳过: 0
FID: 209.2965 (越低越好)
